In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

1. Pivot bronze table so that years are rows and countries are columns

In [0]:
gdp_spark_df = spark.read.table("bronze.gdp_macro_raw")

# 1. Define the list of years to unpivot (2000 to 2019)
years = [str(y) for y in range(2000, 2020)]

# 2. Build a dynamic stack expression to transform year columns into rows
# Format matches your 'YR2000', 'YR2001' column naming pattern
stack_expr = (
    f"stack({len(years)}, "
    + ", ".join([f"'{y}', `YR{y}`, {y}" for y in years])
    + ") as (Year_Str, GDP_Growth_Annual_Pct, Year)"
)

# 3. Apply the stack to unpivot the dataframe
unpivoted_df = gdp_spark_df.select(
    F.col("economy").alias("Country_Code"), F.expr(stack_expr)
).drop("Year_Str")

# 4. Now pivot so that Country Codes become the columns and Year is the row index
gdp_final_spark = (
    unpivoted_df.groupBy("Year")
    .pivot("Country_Code", ["USA", "EMU", "CHN", "JPN", "IND"])
    .agg(F.first("GDP_Growth_Annual_Pct"))
    .orderBy("Year")
)

2. Enforce datatypes

In [0]:
cleaned_silver_df = (
    gdp_final_spark
    .withColumn("Year", F.col("Year").cast(IntegerType()))
    .withColumn("USA", F.col("USA").cast(FloatType()))
    .withColumn("EMU", F.col("EMU").cast(FloatType()))
    .withColumn("CHN", F.col("CHN").cast(FloatType()))
    .withColumn("JPN", F.col("JPN").cast(FloatType()))
    .withColumn("IND", F.col("IND").cast(FloatType()))
)

3. Null check

In [0]:
# Check for any null counts across countries

null_check = cleaned_silver_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cleaned_silver_df.columns]
)
display(null_check)

4. Write table to silver

In [0]:
(
    cleaned_silver_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("silver.gdp_macro_cleaned")
)

print("Silver layer macro table successfully written!")